In [1]:
import os


path_depth = "../../../"  # adjust the current working directory
if "__file__" not in globals():  # check if running in Jupyter Notebook
    # os.system("jupyter nbconvert --to script Controller.ipynb --output Controller")  # convert notebook to script
    os.system("pyuic5 -x View.ui -o View.py")  # convert UI file to Python script

In [2]:
from View import Ui_MainWindow

from PyQt5.QtCore import QStringListModel, QTimer, Qt
from PyQt5.QtGui import QImage, QPixmap, QIcon
from PyQt5.QtWidgets import QMainWindow, QMessageBox, QApplication, QLineEdit

import cv2
import numpy as np
import pickle

In [3]:
database = pickle.load(open(path_depth + "database.pkl", "rb"))

In [4]:
cap = cv2.VideoCapture(0)


class Window(Ui_MainWindow, QMainWindow):
    def __init__(self):
        super().__init__()
        self.setupUi(self)
        self.setWindowFlags(self.windowFlags() | Qt.WindowStaysOnTopHint)
        self.setWindowIcon(QIcon(f"{path_depth}resource/asset/itc_logo.png"))
        self.setWindowTitle("Add New Database Form")

        # setup real-time updates
        self.timer = QTimer(self)
        self.timer.timeout.connect(self.update)
        self.timer.start(33)  # 30 FPS

        self.SKIP_FRAMES = 10
        self.frame_count = 0

        self.show()

    def paintEvent(self, event):
        _, frame = cap.read()

        # self.frame_count += 1
        # if self.frame_count % self.SKIP_FRAMES == 0:
        #     self.frame_count = 0

        # display camera
        tmp_screen = np.array(cv2.resize(frame, dsize=(640, 480), interpolation=cv2.INTER_CUBIC))
        image = cv2.cvtColor(tmp_screen, cv2.COLOR_BGR2RGB)
        q_image = QImage(image.data, 640, 480, QImage.Format.Format_RGB888)
        q_pixmap = QPixmap.fromImage(q_image)
        self.label_camera.setPixmap(q_pixmap)

In [5]:
cap = cv2.VideoCapture(0)
app = QApplication([])

win = Window()


def f_add():
    name = win.lineEdit_name.text()

    if name.__len__() >= 6:
        _, _frame = cap.read()
        _frame = cv2.flip(_frame, 1)
        _frame = cv2.resize(_frame, dsize=(640, 480))

        database.append([name, _frame])
        pickle.dump(database, open(path_depth + "database.pkl", "wb"))


win.pushButton_add_new.clicked.connect(f_add)


app.exec_()


cap.release()

app = None

In [6]:
[[name] for name, img in database]

[['MUY SENGLY'], ['Mr. ABCDEF']]